# Matching scRNA cells to scDNA clones with clonealign

clonealign assigns each cell of a single-cell RNA sequencing (scRNA) data set to one of the clones found by single-cell DNA sequencing (scDNA) of the same tumor (Campbell et al., 2019).
It assumes that the expression of a gene in a cell is proportional to the copy number of the gene in the clone of the cell.
In this tutorial we load a pregrown tumor, sequence its cells with scDNA and scRNA, find the clones of the scDNA, assign the scRNA cells to them with clonealign and score the assignments against the answer key.
clonealign runs from its own conda environment, `iscc-clonealign`, which contains R, TensorFlow and the clonealign package.
The notebook runs in about fifteen minutes on a laptop.

In [1]:
%matplotlib inline
import os, tempfile

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage

from iscc import pregrown
from iscc.sample import Resection
from iscc.data import scDNA, scRNA
from iscc.integrations import biological_types

os.chdir(tempfile.mkdtemp())        # the data and the outputs of the tool go to a scratch directory

## Load a tumor

We load `cin`, one of the [pregrown tumors](../reference/index.md#pregrown-tumors), a chromosomally unstable tumor of about 200,000 cancer cells on a 1 cm field, in which every cancer cell has a doubled genome and the clones differ mostly in copy number.
Its genome has one segment per chromosome arm, and it carries the expression layer of the `iscc` paper, with gene programs, the epithelium as the cell of origin and the per-gene dosage that the paper fitted to TCGA breast tumors.
Under this dosage law, the expression of a gene at total copy number C is its baseline times 1 + s (C/2 - 1), where the sensitivity s of each gene is drawn from a normal distribution with mean 0.86 and standard deviation 0.61 and clipped to the interval from 0 to 1, and copy numbers above 8 count as 8.
A gene with a sensitivity of 1 follows proportional dosage, the assumption of clonealign, and the expression of a gene with a sensitivity of 0 does not change with its copy number.

In [2]:
tumor = pregrown.cin()
print(f"{tumor.get_cancer_size():,} cancer cells of {tumor.get_tumor_size():,} cells after {tumor.time:.0f} generations")

sensitivity = tumor.programs.dosage_sensitivity
print(f"genes with proportional dosage: {np.mean(sensitivity == 1):.0%}, "
      f"genes whose expression ignores copy number: {np.mean(sensitivity == 0):.0%}")

200,156 cancer cells of 465,747 cells after 228 generations
genes with proportional dosage: 41%, genes whose expression ignores copy number: 7%


## Sequence the cells with scDNA and scRNA

As in [the HMMcopy tutorial](tool_hmmcopy.ipynb), we dissociate the specimen, enrich the suspension for epithelial cells by keeping 5% of the stromal and immune cells, and sequence 400 cells with the DLP+ preset of the [`scDNA`](https://pedrofale.github.io/iscc/reference/scDNA/) assay.
We measure the cancer cells of the same suspension with the 10x preset of the [`scRNA`](https://pedrofale.github.io/iscc/reference/scRNA/) assay.
A study keeps the cancer cells after annotating the cell types, and here we take the cell types from the answer key.

In [3]:
specimen = Resection(tumor)
cells = specimen.dissociate(max_cells=3000, layers=["scdna", "scrna"],
                            recovery_probs={"stromal": 0.05, "immune": 0.05}, seed=0)
cell_type = pd.Series(biological_types(cells["cell_type"]["cell_id"]), index=cells["cell_type"].index)

dna = scDNA(n_cells=400, platform="dlp", doublet_rate=0.0, seed=1).run(cells)
rna = scRNA(protocol="10x", seed=2).run(cells, cell_subset=cell_type.index[cell_type == "cancer"])
counts = rna.observed_counts                               # cells x genes
print("scDNA cells:", cell_type[dna.cells].value_counts().to_dict())
print(f"scRNA: {counts.shape[0]:,} cancer cells, {counts.shape[1]:,} genes, "
      f"{counts.sum(axis=1).median():,.0f} UMIs per cell (median)")

scDNA cells: {'cancer': 351, 'stromal': 29, 'epithelial': 20}
scRNA: 1,188 cancer cells, 6,001 genes, 4,018 UMIs per cell (median)


## Find the clones of the scDNA

A study finds the clones of a scDNA run by calling copy number in every cell, for example with [HMMcopy](tool_hmmcopy.ipynb), and clustering the cells by their copy number profiles.
At DLP+ depth a shortcut is enough here.
We average the read count of each cell over each segment of the genome, which is a chromosome arm, divide it by the mean of the cell and multiply it by two, which gives the copy number of each arm relative to a diploid genome.
We cluster the cells by average linkage on the sum of the absolute differences between these profiles, and cut the tree at a distance of 5 copies.
We keep the clusters of at least 10 cells, except the cluster whose median profile is flat, which contains the normal cells.
clonealign normalizes the expected expression of a clone over the genes, so it needs copy number only up to a factor per clone, and we give it the mean relative copy number of each clone.

In [4]:
segment = dna.to_anndata().var["segment"]                  # the segment of every gene
depth = dna.coverage.T.groupby(segment.values).mean().T    # cells x segments
relative = 2 * depth.div(depth.mean(axis=1), axis=0)
tree = linkage(relative.values, method="average", metric="cityblock")
cluster = pd.Series(fcluster(tree, t=5, criterion="distance"), index=relative.index)
size = cluster.value_counts()
flat = relative.groupby(cluster).median().round().eq(2).all(axis=1)
found = size.index[(size >= 10) & ~flat[size.index].to_numpy()]
clones = relative.groupby(cluster).mean().loc[found]
clones.index = [f"dna_{i}" for i in range(1, len(found) + 1)]
clones.round(1).assign(cells=size[found].to_numpy())

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,cells
dna_1,2.0,1.1,2.1,1.0,1.1,2.2,1.0,1.1,2.1,3.2,...,1.6,2.3,2.1,2.1,2.1,3.2,2.1,2.1,1.0,88
dna_2,2.1,1.0,2.1,1.0,1.0,2.1,1.0,1.0,2.0,3.7,...,1.6,2.1,2.4,2.1,2.1,3.1,1.9,2.1,1.0,24
dna_3,2.2,1.1,2.1,1.1,1.1,2.2,1.1,0.6,2.1,3.8,...,1.6,2.1,2.2,2.2,2.2,3.1,2.6,2.1,1.1,23
dna_4,2.0,1.1,2.1,1.2,1.1,2.2,1.0,1.0,2.1,3.5,...,1.6,2.1,2.2,2.2,2.1,3.0,2.0,2.1,1.1,22
dna_5,2.1,1.0,2.1,1.6,1.0,2.1,1.0,1.0,2.2,3.8,...,1.6,2.1,2.1,2.1,1.6,3.0,2.0,2.1,1.0,20
dna_6,2.2,1.1,2.1,1.1,1.1,2.2,1.1,1.0,1.8,3.2,...,1.6,2.2,2.2,2.2,2.2,3.1,2.1,2.1,1.1,17
dna_7,2.1,1.1,2.1,1.1,1.1,2.2,1.1,1.1,2.0,3.8,...,1.6,2.1,2.1,2.2,2.1,3.1,2.2,2.1,1.2,16
dna_8,2.1,1.1,2.1,1.1,1.0,2.1,1.1,0.5,2.0,3.7,...,1.6,2.2,2.1,2.1,2.1,3.1,2.5,2.1,1.0,13
dna_9,2.2,1.1,2.2,1.2,1.1,2.2,1.1,1.1,2.1,3.8,...,1.7,2.2,2.2,2.2,2.2,3.1,2.1,2.2,1.1,13
dna_10,2.1,1.1,2.1,1.1,1.0,2.2,1.1,1.1,2.1,3.7,...,1.7,2.1,2.1,2.2,2.2,3.1,2.0,2.1,1.0,12


We found 11 clones of at least 10 cells among the cancer cells of the scDNA.
They share most of their copy number changes and differ from one another on a few arms.

## The answer key

The answer key holds the true copy number of every scRNA cell.
We put it on the same relative scale and take, for each cell, the scDNA clone whose profile is closest to its own, by the sum of the absolute differences over the arms that clonealign reads.
The likelihood of clonealign needs at least one copy in every clone, so it reads only the arms on which no clone has lost every copy.
The closest clone is the one a perfect assignment would choose.
Not every cell has the profile of a clone found among 400 scDNA cells, and the distance to the closest clone measures how far each cell is from it.

In [5]:
true_cn = cells["cell_cnv"].loc[counts.index]
true_cn = true_cn.T.groupby(segment[true_cn.columns].values).first().T
true_relative = 2 * true_cn.div(true_cn.mean(axis=1), axis=0)
kept = clones.columns[(clones > 0.5).all()]                # the arms on which every clone has a copy
distance = pd.DataFrame(np.abs(true_relative[kept].values[:, None, :] - clones[kept].values[None]).sum(axis=2),
                        index=true_relative.index, columns=clones.index)
closest = distance.idxmin(axis=1)
print(f"distinct true profiles among the scRNA cells: {len(true_relative.round(2).drop_duplicates())}")
pd.DataFrame({"cells": closest.value_counts(),
              "median distance": distance.min(axis=1).groupby(closest).median().round(2)})

distinct true profiles among the scRNA cells: 1008


,cells,median distance
dna_1,453,3.38
dna_10,47,3.14
dna_11,38,2.54
dna_2,90,2.82
dna_3,74,3.34
dna_4,120,3.13
dna_5,87,3.16
dna_6,58,2.65
dna_7,68,3.46
dna_8,69,2.97


The 1,188 scRNA cells have 1,008 distinct copy number profiles, as expected in a chromosomally unstable tumor.
The cells closest to each clone differ from it by a median of 2.5 to 3.5 copies summed over the arms, so the clones of the scDNA describe the copy number of the cells only roughly.

## Write the inputs of clonealign

clonealign reads a cells by genes count matrix and a genes by clones copy number matrix.
We write the counts with the writer of the assay and give every gene on the arms that clonealign reads the copy number of its arm in each clone.

In [6]:
rna.write("clonealign", write_h5ad=False)                  # clonealign/umis.csv, cells x genes
gene_segment = segment[counts.columns]
genes = gene_segment.index[gene_segment.isin(kept)]
copy_number = clones.T.loc[gene_segment[genes]].set_axis(genes)   # genes x clones
copy_number.to_csv("clonealign/copy_number.csv")
print(f"{len(genes):,} genes on {len(kept)} of {clones.shape[1]} arms")

6,001 genes on 39 of 39 arms


## Run clonealign

We run clonealign as its documentation does, filtering the genes and cells with `preprocess_for_clonealign` and fitting with `run_clonealign`, which restarts the variational inference from several initializations, all at their defaults.
A cell whose largest assignment probability is below 0.95 stays unassigned.

In [7]:
%%writefile clonealign/clonealign.R
# clonealign (Campbell et al. 2019) at its defaults
# the Python with TensorFlow of this environment, quiet TensorFlow logs, no terminal control codes
Sys.setenv(RETICULATE_PYTHON = file.path(dirname(dirname(R.home())), "bin", "python"),
           TF_CPP_MIN_LOG_LEVEL = "3", TERM = "dumb")
suppressPackageStartupMessages(library(clonealign))
Y <- as.matrix(read.csv("clonealign/umis.csv", row.names = 1, check.names = FALSE))
L <- as.matrix(read.csv("clonealign/copy_number.csv", row.names = 1, check.names = FALSE))
pp <- preprocess_for_clonealign(Y[, rownames(L)], L)
cat(sprintf("preprocess_for_clonealign kept %d cells and %d genes\n",
            length(pp$retained_cells), length(pp$retained_genes)))

tensorflow::set_random_seed(1)
warned <- character()
fit <- withCallingHandlers(
  run_clonealign(pp$gene_expression_data, pp$copy_number_data, print_elbos = FALSE, verbose = FALSE),
  warning = function(w) {
    warned <<- union(warned, trimws(conditionMessage(w)))
    invokeRestart("muffleWarning")
  })
if (length(warned)) cat("clonealign warned:", warned, sep = "\n")
write.csv(data.frame(cell = pp$retained_cells, clone = fit$clone), "clonealign/clones.csv", row.names = FALSE)

Writing clonealign/clonealign.R


In [8]:
!~/miniconda3/envs/iscc-clonealign/bin/Rscript clonealign/clonealign.R

preprocess_for_clonealign kept 1188 cells and 2816 genes


clonealign warned:
Less than 75% of genes positively correlated with expression - assignment may have failed


## Score the assignments

We count a cell as correctly assigned when clonealign assigns it to its closest scDNA clone.
Assigning every cell to the clone that is closest for most cells gives the baseline.

In [9]:
assigned = pd.read_csv("clonealign/clones.csv", index_col="cell")["clone"].reindex(closest.index).fillna("not kept")
print(f"cells assigned to a clone: {assigned.isin(clones.index).mean():.0%}")
print(f"cells assigned to their closest clone: {(assigned == closest).mean():.0%}")
print(f"baseline, every cell to {closest.value_counts().index[0]}: {closest.value_counts(normalize=True).iloc[0]:.0%}")
pd.crosstab(closest.rename("closest clone"), assigned.rename("clonealign"))

cells assigned to a clone: 65%
cells assigned to their closest clone: 40%
baseline, every cell to dna_1: 38%


clonealign,dna_1,dna_10,dna_11,dna_2,dna_3,dna_4,dna_5,dna_6,dna_7,dna_8,dna_9,unassigned
closest clone,,,,,,,,,,,,
dna_1,319,5,3,1,0,2,0,11,3,0,1,108
dna_10,11,18,1,0,0,1,0,1,0,0,0,15
dna_11,1,2,16,1,0,0,0,0,0,0,0,18
dna_2,34,1,7,19,0,0,0,3,0,0,0,26
dna_3,21,4,0,2,2,9,1,2,3,0,0,30
dna_4,48,0,4,0,0,5,1,4,4,0,0,54
dna_5,11,0,3,0,0,9,31,2,0,0,0,31
dna_6,12,0,1,0,0,0,0,29,0,0,1,15
dna_7,13,1,0,0,0,0,0,2,16,0,0,36


clonealign assigned 65% of the cells to a clone, and 40% of all cells to their closest clone, close to the 38% of assigning every cell to dna_1.
It also warned that fewer than 75% of the genes are positively correlated with expression, so that the assignment may have failed.
It assigned 319 of the 453 cells closest to dna_1 to that clone, and at most half of the cells closest to each of the other clones to theirs.
clonealign assumes that expression is proportional to copy number, while here 59% of the genes are partly or fully buffered, gene programs move expression further and the cells differ from their clones on several arms, so the expression differences between clones may be smaller than the model expects.
The `iscc` paper reports the same pattern for an assignment under proportional dosage, which was correct for fewer cells on the expression of `iscc` than on expression generated under proportional dosage, and least so in tumors with five or six copy number profiles.

## Make the benchmark harder or easier

The settings of the tumor and of the assays change how hard this benchmark is, and [Shape the tumor](02_shape_the_tumor.ipynb) and [Shape the data](03_shape_the_data.ipynb) show their effects.
The settings of the tumor take effect when we grow a tumor from a configuration instead of loading a pregrown one.

- **Dosage.** A `dosage_sensitivity_mean` of 1 with a `dosage_sensitivity_sd` of 0 gives every gene proportional dosage, the assumption of clonealign, and lower values buffer more genes. The gene programs (`program_params`) and the microenvironment (`microenv_params`) of the expression layer move expression further away from copy number.
- **Clones.** The copy number rate of the cancer cells (`cnv_prob` in `cell_params`) sets how many clones differ in copy number and on how many arms.
- **scRNA.** The depth (`mu_lib`), the overdispersion (`dispersion`), the ambient RNA (`ambient_frac`), the doublets (`doublet_rate`) and the dropout (`dropout_mid`) of the assay set the noise around the dosage signal.
- **scDNA.** Fewer cells or a shallower depth find fewer clones, a lower or higher cut of the tree finds more or fewer clones, and clones called with [HMMcopy](tool_hmmcopy.ipynb) carry the errors of the caller into the copy number that clonealign reads.

The [`scRNA`](https://pedrofale.github.io/iscc/reference/scRNA/), [`scDNA`](https://pedrofale.github.io/iscc/reference/scDNA/) and [`GenotypeTumor`](https://pedrofale.github.io/iscc/reference/GenotypeTumor/) pages document every parameter.